In [2]:
%%configure -f
{
    "jars": ["https://repo1.maven.org/maven2/org/datasyslab/geotools-wrapper/1.9.1-33.5/geotools-wrapper-1.9.1-33.5.jar", "https://repo1.maven.org/maven2/org/apache/sedona/sedona-spark-shaded-4.1_2.13/1.9.1/sedona-spark-shaded-4.1_2.13-1.9.1.jar"]
}

StatementMeta(, db3b8340-e744-4ac3-81ee-b8f7d4e2782c, -1, Finished, Available, Finished, True)

In [3]:
%run manage_files

StatementMeta(, db3b8340-e744-4ac3-81ee-b8f7d4e2782c, 11, Finished, Available, Finished, True)

In [4]:
%run manage_tables

StatementMeta(, db3b8340-e744-4ac3-81ee-b8f7d4e2782c, 17, Finished, Available, Finished, True)

In [5]:
%run transformation

StatementMeta(, db3b8340-e744-4ac3-81ee-b8f7d4e2782c, 26, Finished, Available, Finished, True)

In [6]:
from pyspark.sql.functions import *
from sedona.spark import SedonaContext

StatementMeta(, db3b8340-e744-4ac3-81ee-b8f7d4e2782c, 27, Finished, Available, Finished, False)

In [7]:
YELLOW_TAXI_FOLDER_PATH = "Files/NYC_Taxi/yellow_tripdata"

YELLOW_TAXI_SCHEMA = {
    "pickup_datetime": "timestamp",
    "dropoff_datetime": "timestamp",
    "passenger_count": "integer",
    "trip_distance": "double",
    "pickup_location_id": "integer",
    "dropoff_location_id": "integer",
    "pickup_longitude": "double",
    "pickup_latitude": "double",
    "dropoff_longitude": "double",
    "dropoff_latitude": "double",
    "fare_amount": "double",
    "total_amount": "double",
    "tip_amount": "double"
}

YELLOW_TAXI_COLUMN_MAPPING = {
    "Trip_Pickup_DateTime": "pickup_datetime",
    "tpep_pickup_datetime": "pickup_datetime",
    "Trip_Dropoff_DateTime": "dropoff_datetime",
    "tpep_dropoff_datetime": "dropoff_datetime",
    "Passenger_Count": "passenger_count",
    "Trip_Distance": "trip_distance",
    "PULocationID": "pickup_location_id",
    "DOLocationID": "dropoff_location_id",
    "Start_Lon": "pickup_longitude",
    "Start_Lat": "pickup_latitude",
    "End_Lon": "dropoff_longitude",
    "End_Lat": "dropoff_latitude",
    "Fare_Amt": "fare_amount",
    "Total_Amt": "total_amount",
    "Tip_Amt": "tip_amount"
}

StatementMeta(, db3b8340-e744-4ac3-81ee-b8f7d4e2782c, 28, Finished, Available, Finished, False)

In [8]:
def get_existing_months(df, date_column):
    return [row["yyyy-mm"] for row in df.select(date_format(date_column, "yyyy-MM").alias("yyyy-mm")).distinct().collect()]

def get_loaded_months(table_name, date_column):
    if not is_table_exist(table_name, "Silver_lakehouse"):
        return []
    df = read_table(table_name, "Silver_lakehouse")
    return get_existing_months(df, date_column)

def get_new_files(file_paths, existing_months):
    return [path for path in file_paths if path[-15:-8] not in existing_months]

def get_new_parquet_files(folder_path, table_name, date_column):
    file_paths = get_parquet_files(folder_path)
    existing_months = get_loaded_months(table_name, date_column)
    return get_new_files(file_paths, existing_months)

StatementMeta(, db3b8340-e744-4ac3-81ee-b8f7d4e2782c, 29, Finished, Available, Finished, False)

In [13]:
TAXI_ZONES_SHAPEFILE_FOLDER_PATH = "Files/NYC_Taxi/taxi_zones/"

def create_sedona_context(spark):
    return SedonaContext.create(spark)

def read_taxi_zones_shapefile(sedona, file_path):
    return sedona.read.format("org.apache.sedona.sql.datasources.shapefile.ShapefileDataSource").load(file_path)

def create_temp_view(df, view_name):
    df.createOrReplaceTempView(view_name)

def transform_taxi_zones(df):
    return df.selectExpr("*", "ST_Transform(geometry, 'EPSG:2263', 'EPSG:4326') AS new_geometry")

def enrich_taxi_locations(df, sedona):
    create_temp_view(df, "trips")
    return sedona.sql("""
        SELECT
            trips.pickup_datetime,
            trips.dropoff_datetime,
            trips.passenger_count,
            trips.trip_distance,
            COALESCE(trips.pickup_location_id, pickup_zone.LocationID) AS pickup_location_id,
            COALESCE(trips.dropoff_location_id, dropoff_zone.LocationID) AS dropoff_location_id,
            trips.fare_amount,
            trips.total_amount,
            trips.tip_amount
        FROM trips
        LEFT JOIN zones AS pickup_zone 
        ON ST_Contains(pickup_zone.new_geometry, ST_Point(trips.pickup_longitude, trips.pickup_latitude))
        LEFT JOIN zones AS dropoff_zone
        ON ST_Contains(dropoff_zone.new_geometry, ST_Point(trips.dropoff_longitude, trips.dropoff_latitude))
    """)

def prepare_taxi_zones(shapefile_folder_path):
    sedona = create_sedona_context(spark)
    zones_df = read_taxi_zones_shapefile(sedona, shapefile_folder_path)
    zones_df = transform_taxi_zones(zones_df)
    create_temp_view(zones_df, "zones")
    return sedona

StatementMeta(, db3b8340-e744-4ac3-81ee-b8f7d4e2782c, 35, Finished, Available, Finished, False)

In [10]:
def normalize_taxi(df, column_mapping, schema):
    df = rename_columns(df, column_mapping)
    df = add_missing_columns(df, schema)
    df = cast_columns(df, schema)
    df = select_columns(df, schema)
    return df

StatementMeta(, db3b8340-e744-4ac3-81ee-b8f7d4e2782c, 31, Finished, Available, Finished, False)

In [15]:
def process(folder_path, table_name, date_column):
    new_files = get_new_parquet_files(folder_path, table_name, date_column)
    sedona = prepare_taxi_zones(TAXI_ZONES_SHAPEFILE_FOLDER_PATH)
    for file_path in new_files[:1]:
        df = read_parquet_file(file_path)
        df = normalize_taxi(df, YELLOW_TAXI_COLUMN_MAPPING, YELLOW_TAXI_SCHEMA)
        df = enrich_taxi_locations(df, sedona)
        save_table(df, table_name, "Silver_lakehouse", "append")


StatementMeta(, db3b8340-e744-4ac3-81ee-b8f7d4e2782c, 37, Finished, Available, Finished, False)

In [16]:
process(
    "Files/NYC_Taxi/yellow_tripdata",
    "yellow_tripdata",
    "pickup_datetime"
)

StatementMeta(, db3b8340-e744-4ac3-81ee-b8f7d4e2782c, 38, Submitted, Running, Running, True)